%md
# Marktdata data loader
Deze code doet het volgende:

1. **FedML importeren/installeren:** Probeert de `fedml_databricks`-module te importeren. Als deze niet aanwezig is, wordt deze automatisch geïnstalleerd via pip.
2. **Helperfunctie:** Definieert een functie `dsp_to_pandas` die de uitvoer van een query omzet naar een pandas DataFrame, ongeacht het oorspronkelijke formaat (tuple, list, dict).
3. **Verbindingsgegevens ophalen:** Haalt de benodigde connectiegegevens (adres, poort, gebruiker, wachtwoord, schema) op uit Databricks secrets.
4. **Viewnaam ophalen:** Maakt een widget aan voor de viewnaam (`DSP_VIEW`) en leest de waarde uit.
5. **FedML-configuratie:** Stelt een configuratieobject samen met de connectieparameters.
6. **Verbinding maken en data ophalen:** 
   - Maakt verbinding met de Datasphere-database via FedML.
   - Voert een SQL-query uit op de opgegeven view (maximaal 5 rijen).
   - Zet het resultaat om naar een pandas DataFrame.
   - Print een melding of er data is opgehaald of niet.
   - Vangt eventuele fouten af en toont een foutmelding.

Kortom: deze code haalt (maximaal 5) rijen op uit een opgegeven Datasphere-view en zet deze om naar een pandas DataFrame in Databricks.

In [0]:
import sys, subprocess
import pandas as pd

# 1) FedML import / install
try:
    from fedml_databricks import DbConnection  # type: ignore
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "fedml-databricks==1.0.5"])
    from fedml_databricks import DbConnection  # type: ignore

# 2) Helper: FedML output → pandas DataFrame
def dsp_to_pandas(result):
    if isinstance(result, tuple):
        result = result[0]
    if hasattr(result, "columns"):
        return result
    if isinstance(result, list):
        return pd.DataFrame(result)
    if isinstance(result, dict):
        return pd.DataFrame([result])
    return pd.DataFrame(result)

# 3) Connection details - geladen uit Databricks secrets
DSP_ADDRESS = dbutils.secrets.get(scope="Datasphere", key="address")
DSP_PORT = int(dbutils.secrets.get(scope="Datasphere", key="port"))
DSP_USER = dbutils.secrets.get(scope="Datasphere", key="user")
DSP_PASSWORD = dbutils.secrets.get(scope="Datasphere", key="password")

# Datasphere Space (= HANA schema)
DSP_SCHEMA = dbutils.secrets.get(scope="Datasphere", key="schema")

# View / model
dbutils.widgets.text("DSP_VIEW", "")
DSP_VIEW = dbutils.widgets.get("DSP_VIEW")

# 4) FedML config
config_extended = {
    "address": DSP_ADDRESS,
    "port": DSP_PORT,
    "user": DSP_USER,
    "password": DSP_PASSWORD,
    "schema": "",
    "encrypt": True,
    "sslValidateCertificate": "true",
    "disableCloudRedirect": "false",
    "communicationTimeout": 60000,
    "autocommit": True,
    "sslUseDefaultTrustStore": "true",
}

# 5) Verbinding + data ophalen
try:
    dsp = DbConnection(dict_obj=config_extended)

    sql_head = f'SELECT * FROM "{DSP_SCHEMA}"."{DSP_VIEW}" LIMIT 5'
    result = dsp.execute_query(sql_head)
    rows_df = dsp_to_pandas(result)

    if rows_df.empty:
        print("⚠️ Query uitgevoerd, maar geen data teruggekregen.")
    else:
        print(f"✅ Data succesvol opgehaald.")

except Exception as e:
    print("❌ Fout bij ophalen van data uit Datasphere:")
    raise e


2026-06-18 13:32:44,925: fedml_databricks.logger INFO: Attempting DSP check.
2026-06-18 13:32:44,935: fedml_databricks.logger INFO: DSP check completed. Connected to DSP.
✅ Data succesvol opgehaald.


Controle IP adress (poort 0 t/m 32 zijn gewhitelist)

In [0]:
# controle IP adress, de poort (laatste 2 cijfers) moet in de range 0 t/m 32 zitten. 
import requests
print(requests.get("https://api.ipify.org").text)


3.77.216.216


# Inladen naar tabel in Databricks
Deze notebook laadt data uit een externe bron (via `dsp.execute_query`) batchgewijs in een Databricks Delta-tabel. 

**Stappen:**
1. **Helpers:** Functies om query-resultaten om te zetten naar een pandas DataFrame met juiste kolomnamen, en om pandas DataFrames om te zetten naar Spark DataFrames met een stabiel schema.
2. **Instellingen:** Doeltabel, batchgrootte en optioneel een sorteersleutel worden ingesteld.
3. **Totaal aantal rijen ophalen:** Het totaal aantal rijen in de bronview wordt opgehaald via een SQL COUNT-query.
4. **Batchgewijs laden en schrijven:** 
   - Data wordt in batches opgehaald uit de bronview.
   - Elke batch wordt omgezet naar een Spark DataFrame en in de doeltabel geschreven (eerste batch met 'overwrite', daarna met 'append').
   - De voortgang wordt geprint.
5. **Afronding:** Na het laden van alle data wordt een bevestiging geprint dat de tabel is opgeslagen.

Dit proces zorgt ervoor dat grote datasets efficiënt en betrouwbaar in Databricks worden geladen.

In [0]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

# ---------- Helpers ----------
def fedml_result_to_pandas(result):
    """FedML execute_query result -> pandas DF met kolomnamen (uit metadata)."""
    if not isinstance(result, tuple) or len(result) < 2:
        raise ValueError("execute_query returned no (rows, cols) tuple")

    rows = result[0]
    cols = result[1]

    # kolomnamen bepalen
    if isinstance(cols, list) and len(cols) > 0:
        if isinstance(cols[0], str):
            colnames = cols
        elif isinstance(cols[0], dict) and "name" in cols[0]:
            colnames = [c["name"] for c in cols]
        else:
            # fallback: eerste value uit dict
            colnames = [list(c.values())[0] for c in cols]
    else:
        colnames = None

    return pd.DataFrame(rows, columns=colnames)

def pandas_to_spark_stable(pdf: pd.DataFrame):
    """Maak schema stabiel (geen kolommen die verdwijnen): cast alles naar string."""
    pdf = pdf.where(pd.notnull(pdf), None).astype("string")
    return spark.createDataFrame(pdf)

# ---------- Instellingen ----------
# Doeltabel (Unity Catalog)
table_fqn = f"workspace.default.{DSP_VIEW}"
batch_size = 50_000                             # pas aan (10k/50k/100k) afhankelijk van grootte/snelheid
order_by = None

# ---------- 1) Bepaal watermark op basis van CREATIE_DAT ----------
table_exists = spark.catalog.tableExists(table_fqn)

if table_exists:
    existing_cols = [f.name.upper() for f in spark.table(table_fqn).schema.fields]
    has_watermark_col = "CREATIE_DAT" in existing_cols
else:
    has_watermark_col = False

if table_exists and has_watermark_col:
    max_date = spark.table(table_fqn).agg(F.max("CREATIE_DAT")).collect()[0][0]
    print(f"✅ Tabel bestaat. Hoogste CREATIE_DAT in Delta: {max_date}")
    where_clause = f' WHERE "CREATIE_DAT" > \'{max_date}\'' if max_date else ""
elif table_exists and not has_watermark_col:
    max_date = None
    where_clause = ""
    print("ℹ️ Tabel bestaat maar CREATIE_DAT ontbreekt nog — volledige herload om kolom toe te voegen.")
else:
    max_date = None
    where_clause = ""
    print("ℹ️ Tabel bestaat nog niet — volledige initiële load.")

# ---------- 2) Aantal nieuwe rijen ophalen ----------
sql_count = f'SELECT COUNT(*) AS CNT FROM "{DSP_SCHEMA}"."{DSP_VIEW}"{where_clause}'
cnt_res = dsp.execute_query(sql_count)
cnt_df = fedml_result_to_pandas(cnt_res)
total = int(cnt_df.iloc[0, 0])

print(f"✅ Aantal nieuwe rijen om te laden: {total:,}")

if total == 0:
    print("ℹ️ Geen nieuwe rijen gevonden. Niets te laden.")
else:
    # ---------- 3) Batch-load + write ----------
    written = 0
    offset = 0
    # overwrite bij initiële load of wanneer CREATIE_DAT nog niet in de tabel zit
    first_write = not table_exists or not has_watermark_col

    while True:
        order_clause = f' ORDER BY "{order_by}"' if order_by else ""
        sql_batch = (
            f'SELECT * FROM "{DSP_SCHEMA}"."{DSP_VIEW}"'
            f'{where_clause}'
            f'{order_clause} '
            f'LIMIT {batch_size} OFFSET {offset}'
        )

        res = dsp.execute_query(sql_batch)
        pdf = fedml_result_to_pandas(res)

        if pdf.empty:
            break

        sdf = pandas_to_spark_stable(pdf)

        mode = "overwrite" if first_write else "append"
        (sdf.write
            .format("delta")
            .mode(mode)
            .option("overwriteSchema", "true" if first_write else "false")
            .saveAsTable(table_fqn)
        )

        batch_rows = len(pdf)
        written += batch_rows
        offset += batch_rows
        first_write = False

        print(f"✅ Batch geschreven: {batch_rows:,} rijen | Totaal geschreven: {written:,}/{total:,}")

    print(f"🎉 Klaar! {written:,} nieuwe rijen toegevoegd aan: {table_fqn}")


✅ Tabel bestaat. Hoogste CREATIE_DAT in Delta: 2026-06-17 13:00:21.038966
✅ Aantal nieuwe rijen om te laden: 1
✅ Batch geschreven: 1 rijen | Totaal geschreven: 1/1
🎉 Klaar! 1 nieuwe rijen toegevoegd aan: workspace.default.4V_WorkedHours
